In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
with_ppg=[]
import os
for dirname, _, filenames in os.walk('/Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/Challenge2015_csv/'):
    for filename in filenames:
        if(filename!="ALARMS"):
            if("PLETH" in pd.read_csv(dirname+filename).columns):
                with_ppg.append(filename[0:-4])

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

df=pd.read_csv("/Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/Challenge2015_csv/ALARMS", names=["Sample", "Label", "TrueAlarm"])
df_relevant=df.loc[df["Label"]!="Asystole"].copy()
df_relevant

df_filtered=df_relevant.loc[df_relevant["Sample"].isin(with_ppg)].copy()

df_filtered["Path"]=["/Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/Challenge2015_csv/"+x+".csv" for x in df_filtered["Sample"]]
df_filtered

,Sample,Label,TrueAlarm,Path
0,v100s,Ventricular_Tachycardia,0,/Users/sarahalabdulrazzak/Desktop/Capstone/mye...
1,v101l,Ventricular_Tachycardia,0,/Users/sarahalabdulrazzak/Desktop/Capstone/mye...
2,v102s,Ventricular_Tachycardia,0,/Users/sarahalabdulrazzak/Desktop/Capstone/mye...
6,t106s,Tachycardia,1,/Users/sarahalabdulrazzak/Desktop/Capstone/mye...
7,t107l,Tachycardia,1,/Users/sarahalabdulrazzak/Desktop/Capstone/mye...
...,...,...,...,...
740,b840s,Bradycardia,1,/Users/sarahalabdulrazzak/Desktop/Capstone/mye...
743,v843l,Ventricular_Tachycardia,0,/Users/sarahalabdulrazzak/Desktop/Capstone/mye...
746,v846s,Ventricular_Tachycardia,0,/Users/sarahalabdulrazzak/Desktop/Capstone/mye...
748,v848s,Ventricular_Tachycardia,0,/Users/sarahalabdulrazzak/Desktop/Capstone/mye...


In [4]:
afib_paths=[]
healthy_paths=[]
afib_samples=[]
healthy_samples=[]

for dirname, _, filenames in os.walk('/Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/mimic_perform_af_csv'):
    for filename in filenames:
        if("csv" in filename):
            afib_paths.append(dirname+"/"+filename)
            filename_split=filename.split("_")
            afib_samples.append(filename_split[2]+"_"+filename_split[3])

for dirname, _, filenames in os.walk('/Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/mimic_perform_non_af_csv'):
    for filename in filenames:
        if("csv" in filename):
            healthy_paths.append(dirname+"/"+filename)
            filename_split=filename.split("_")
            healthy_samples.append(filename_split[2]+"_"+filename_split[3]+"_"+filename_split[4])


afib_df=pd.DataFrame({"Sample":afib_samples,
        "Label":19*["Atrial_Fibrillation"],
        "TrueAlarm":19*[True],
        "Path":afib_paths})

healthy_df=pd.DataFrame({"Sample":healthy_samples,
        "Label":16*["Healthy"],
        "TrueAlarm":16*[True],
        "Path":healthy_paths})

df_filtered=pd.concat([df_filtered, afib_df, healthy_df])
df_filtered

,Sample,Label,TrueAlarm,Path
0,v100s,Ventricular_Tachycardia,0,/Users/sarahalabdulrazzak/Desktop/Capstone/mye...
1,v101l,Ventricular_Tachycardia,0,/Users/sarahalabdulrazzak/Desktop/Capstone/mye...
2,v102s,Ventricular_Tachycardia,0,/Users/sarahalabdulrazzak/Desktop/Capstone/mye...
6,t106s,Tachycardia,1,/Users/sarahalabdulrazzak/Desktop/Capstone/mye...
7,t107l,Tachycardia,1,/Users/sarahalabdulrazzak/Desktop/Capstone/mye...
...,...,...,...,...
11,non_af_006,Healthy,1,/Users/sarahalabdulrazzak/Desktop/Capstone/mye...
12,non_af_010,Healthy,1,/Users/sarahalabdulrazzak/Desktop/Capstone/mye...
13,non_af_011,Healthy,1,/Users/sarahalabdulrazzak/Desktop/Capstone/mye...
14,non_af_016,Healthy,1,/Users/sarahalabdulrazzak/Desktop/Capstone/mye...


In [5]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

def check_and_resample(file_path):
    if 'Challenge2015' in file_path:
        fs_original = 250
        target_fs = 125
        resample_needed = True
    elif 'mimic' in file_path.lower():
        fs_original = 125
        target_fs = 125
        resample_needed = False
    else:
        raise ValueError("Unknown source for file: " + file_path)
    
    print(f"{file_path}: Original sampling rate = {fs_original} Hz")

    # Load the data
    df = pd.read_csv(file_path)

    if not resample_needed:
        print("  Signal is already at 125 Hz — no resampling needed.")
        return df
    
    # Resample if needed
    print(f"  Resampling from {fs_original} Hz to {target_fs} Hz...")

    if 'PLETH' in df.columns:
        signal = df['PLETH'].values
    else:
        raise ValueError("PLETH column not found in the data.")

    # Create new time points
    original_time = np.arange(len(signal)) / fs_original
    new_time = np.arange(0, original_time[-1], 1/target_fs)

    # Interpolate
    interp_function = interp1d(original_time, signal, kind='linear')
    new_signal = interp_function(new_time)

    # Create a new DataFrame for the resampled signal
    df_resampled = pd.DataFrame(new_signal, columns=['PLETH'])

    print("  Resampling complete.")
    return df_resampled


In [6]:
chunk_len=1024

def standardize(data):
    return ( data - np.mean(data) ) / np.std(data)

def getWaveform(label, index, sampleType, startPoint=0, duration=chunk_len, trueAlarm=1):
    if label in ["Atrial_Fibrillation", "Healthy"]:
        file_path = df_filtered.loc[
            (df_filtered["Label"] == label) & (df_filtered["TrueAlarm"] == trueAlarm), 
            "Path"
        ].values[index]
        
        # First load and resample the file
        df = check_and_resample(file_path)
        
        # Then extract the signal after resampling
        if sampleType in df.columns:
            signal = df[sampleType].values[startPoint:startPoint + int(duration/2)]
            return standardize(signal)
        else:
            raise ValueError(f"{sampleType} column not found in file.")
    
    else:
        if sampleType == "ECG":
            sampleType = "II"
        elif sampleType == "PPG":
            sampleType = "PLETH"
        
        file_path = df_filtered.loc[
            (df_filtered["Label"] == label) & (df_filtered["TrueAlarm"] == trueAlarm), 
            "Path"
        ].values[index]

        try:
            signal = pd.read_csv(file_path)[sampleType].values[startPoint:startPoint + duration]
        except:
            signal = pd.read_csv(file_path)["V"].values[startPoint:startPoint + duration]
        
        return standardize(signal)


In [ ]:
import matplotlib.pyplot as plt
 
plt.plot(getWaveform("Atrial_Fibrillation", 0, "ECG"))

In [ ]:
plt.plot(getWaveform("Atrial_Fibrillation", 0, "PPG"))

In [7]:
def getWaveforms(label, index, sampleType, duration=chunk_len, trueAlarm=1):
    # Get the file path for the specified label, index, and trueAlarm
    file_path = df_filtered.loc[
        (df_filtered["Label"] == label) & (df_filtered["TrueAlarm"] == trueAlarm), 
        "Path"
    ].values[index]
    
    print(f"Loading file: {file_path}")

    # Handle sampleType consistency
    if sampleType == "ECG":
        sampleType = "II"
    elif sampleType == "PPG":
        sampleType = "PLETH"
    
    # Load and resample if necessary
    try:
        # Resample the data if needed
        df = check_and_resample(file_path)
        
        # Print the available columns to help debug if sampleType is not found
        print(f"Available columns: {df.columns.tolist()}")
        
        # If II column isn't found, check for "ECG" column
        if sampleType == "II" and "ECG" in df.columns:
            print("Warning: 'II' column not found. Falling back to 'ECG' column.")
            sampleType = "ECG"  # Use 'ECG' instead of 'II'
        
        # Check if the column exists after fallback
        if sampleType not in df.columns:
            print(f"Warning: {sampleType} column not found in file. Returning empty list.")
            return []
        
        fullWaveform = df[sampleType].values
        print(f"Loaded {len(fullWaveform)} samples from {sampleType}.")
    except Exception as e:
        print(f"Error loading file {file_path}: {e}")
        return []

    # Ensure there is enough data to split into chunks of the specified duration
    if len(fullWaveform) < duration:
        print(f"Warning: Not enough data to create a full chunk of size {duration}.")
        return []

    # Split into periods of specified length (duration)
    periodsList = [
        fullWaveform[i * duration : (i + 1) * duration] 
        for i in range(int(np.floor(len(fullWaveform) / duration)))
    ]
    
    print(f"Created {len(periodsList)} chunks of size {duration}.")
    
    return periodsList


In [ ]:
periods = getWaveforms("Atrial_Fibrillation", 0, "ECG")
if periods:  # Only try to plot if periods are found
    for i in range(min(10, len(periods))):
        plt.plot(periods[i])
        plt.show()
else:
    print("No valid periods found.")


In [15]:
import os
import pandas as pd
import numpy as np

# Define the output folder path
output_folder = "/Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/"

# List of arrhythmia types you want to extract
arrhythmia_types = [
    "Ventricular_Flutter_Fib", "Ventricular_Tachycardia", "Tachycardia", "Bradycardia", 
    "Atrial_Fibrillation", "Healthy"
]

# Helper function to create directories if they don't exist
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

# Helper function to write CSV data to a file
def write_csv(data, output_path):
    data.to_csv(output_path, index=False)
    print(f"Saved to {output_path}")


In [ ]:
for arrhythmia in arrhythmia_types:
    df_filtered_arrhythmia = df_filtered[df_filtered["Label"] == arrhythmia].copy()

    print(f"Processing {arrhythmia} samples. Number of samples: {len(df_filtered_arrhythmia)}")

    for idx in range(len(df_filtered_arrhythmia)):
        row = df_filtered_arrhythmia.iloc[idx]
        print(f"Processing row {idx} - Sample: {row['Sample']}, Path: {row['Path']}")

        try:
            # Extract ECG and PPG signals using the `getWaveform` function
            ecg_signal = getWaveform(arrhythmia, idx, "ECG", trueAlarm=1)
            ppg_signal = getWaveform(arrhythmia, idx, "PPG", trueAlarm=1)

            # Combine ECG and PPG signals into a single DataFrame
            combined_data = pd.DataFrame({
                'ECG': ecg_signal,
                'PPG': ppg_signal
            })

            # Define the patient's file name based on the sample
            patient_file = f"{row['Sample']}.csv"
            patient_output_path = os.path.join(arrhythmia_folder, patient_file)

            # Write the combined ECG and PPG data to CSV
            write_csv(combined_data, patient_output_path)

        except Exception as e:
            print(f"Error processing sample {row['Sample']} for {arrhythmia}: {e}")


Processing Ventricular_Flutter_Fib samples. Number of samples: 47
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Ventricular_Flutter_Fib/f120s.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Ventricular_Flutter_Fib/f121l.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Ventricular_Flutter_Fib/f129l.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Ventricular_Flutter_Fib/f130s.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Ventricular_Flutter_Fib/f137l.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Ventricular_Flutter_Fib/f138s.csv
Error processing sample f144s for Ventricular_Flutter_Fib: index 6 is out of bounds for axis 0 with size 6
Error processing sample f189l for Ventricular_Flutter_Fib: index 7 is out of bounds for axis 0 with size 6


/var/folders/87/1x2ps50d7dx1y569r31__0t00000gn/T/ipykernel_15915/973443539.py:4: RuntimeWarning: invalid value encountered in divide
  return ( data - np.mean(data) ) / np.std(data)


Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Ventricular_Tachycardia/v155l.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Ventricular_Tachycardia/v158s.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Ventricular_Tachycardia/v159l.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Ventricular_Tachycardia/v160s.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Ventricular_Tachycardia/v162s.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Ventricular_Tachycardia/v164s.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Ventricular_Tachycardia/v166s.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Ventricular_Tachycardia/v168s.csv
Saved to /Users/sarahalabdulrazzak/Deskt

/var/folders/87/1x2ps50d7dx1y569r31__0t00000gn/T/ipykernel_15915/973443539.py:4: RuntimeWarning: invalid value encountered in divide
  return ( data - np.mean(data) ) / np.std(data)
/var/folders/87/1x2ps50d7dx1y569r31__0t00000gn/T/ipykernel_15915/973443539.py:4: RuntimeWarning: invalid value encountered in divide
  return ( data - np.mean(data) ) / np.std(data)


Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Tachycardia/t506s.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Tachycardia/t507l.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Tachycardia/t508s.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Tachycardia/t509l.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Tachycardia/t520s.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Tachycardia/t521l.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Tachycardia/t524s.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Tachycardia/t546s.csv
Saved to /Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/arrhythmia_data/folder/Tachycardia/t547l.csv
Saved to /Users/sarahalabdul